# Chapter 7 — Search In-Depth
## Topic 3: Advanced Retrieval — DPR, ColBERT, SPLADE, Learned Sparse

**Run cells in order. Each cell is a self-contained module.**

- Module 1: Setup and shared data
- Module 2: DPR — asymmetric bi-encoder simulation with E5 prefixes
- Module 3: ColBERT — MaxSim late interaction scoring
- Module 4: SPLADE — learned sparse expansion simulation
- Module 5: Head-to-head comparison and upgrade path for your project

**Install:** `pip install sentence-transformers numpy rank-bm25`  
No GPU required for this notebook — all models run on CPU.


## Module 1: Setup and Shared Data

Run this first. Loads the knowledge base and imports used by all modules.

In [1]:
"""
MODULE 1: Setup and Shared Data

Same 6-chunk knowledge base as Topics 1 and 2.
Adds QUERY_VARIATIONS to test vocabulary mismatch cases.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

# -----------------------------------------------------------------------
# Knowledge base: 6 chunks from 4 FD source documents
# -----------------------------------------------------------------------
KNOWLEDGE_BASE = [
    {"id": 0, "text": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.", "source": "01_FD_FAQ.pdf", "doc_type": "faq"},
    {"id": 1, "text": "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.", "source": "04_FD_Policy_Reference.pdf", "doc_type": "policy"},
    {"id": 2, "text": "This SOP covers Fixed Deposit premature withdrawal. Step 1: Verify FD account eligibility. Step 2: Calculate penalty. Step 3: Process premature withdrawal and apply 1 percent penalty. Step 4: Update FD records.", "source": "03_FD_SOPs.pdf", "doc_type": "sop"},
    {"id": 3, "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.", "source": "02_FD_Product_Guide.pdf", "doc_type": "product"},
    {"id": 4, "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.", "source": "02_FD_Product_Guide.pdf", "doc_type": "product"},
    {"id": 5, "text": "Early exit from your deposit account will attract a deduction from your returns.", "source": "04_FD_Policy_Reference.pdf", "doc_type": "policy"},
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]

# Test queries: same intent, different vocabulary/language
QUERY_VARIATIONS = [
    ("English (standard)",          "premature withdrawal penalty FD"),
    ("English (paraphrase)",        "early closure charges fixed deposit"),
    ("Hinglish (maturity)",         "machurity ka paisa kab aayega"),
    ("Hinglish (premature)",        "FD band karna hai penalty kitni hogi"),
    ("Mixed (byaj = interest)",     "byaj wala FD premature nikalna hai"),
]

CORPUS_TEXTS = [doc["text"] for doc in KNOWLEDGE_BASE]

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

print("Knowledge base loaded:")
for doc in KNOWLEDGE_BASE:
    print(f"  Doc {doc['id']} [{doc['doc_type'].upper():<7}]: {doc['text'][:65]}...")
print(f"\nQuery variations: {len(QUERY_VARIATIONS)}")
for label, q in QUERY_VARIATIONS:
    print(f"  [{label:<30}] {q}")
print("\nModule 1 complete. Run Module 2.")



Knowledge base loaded:
  Doc 0 [FAQ    ]: Premature withdrawal of FD incurs a 1 percent penalty on the appl...
  Doc 1 [POLICY ]: Fixed Deposit premature closure is allowed subject to applicable ...
  Doc 2 [SOP    ]: This SOP covers Fixed Deposit premature withdrawal. Step 1: Verif...
  Doc 3 [PRODUCT]: The Fixed Deposit interest rate for 24 months is 7.1 percent per ...
  Doc 4 [PRODUCT]: Senior citizens receive an additional 0.5 percent interest on all...
  Doc 5 [POLICY ]: Early exit from your deposit account will attract a deduction fro...

Query variations: 5
  [English (standard)            ] premature withdrawal penalty FD
  [English (paraphrase)          ] early closure charges fixed deposit
  [Hinglish (maturity)           ] machurity ka paisa kab aayega
  [Hinglish (premature)          ] FD band karna hai penalty kitni hogi
  [Mixed (byaj = interest)       ] byaj wala FD premature nikalna hai

Module 1 complete. Run Module 2.


## Module 2: DPR — Asymmetric Bi-Encoder with E5 Prefixes

DPR uses two separate encoders: one for queries, one for passages.
The practical modern equivalent is E5 with "query:" and "passage:" prefixes.

This module shows:
- What asymmetric encoding is and why it helps
- How to use `paraphrase-multilingual-MiniLM-L12-v2` in symmetric mode (current project)
- How to use E5-style prefixes to simulate asymmetric encoding
- The difference in similarity scores

**Note:** For a true DPR comparison you need `intfloat/multilingual-e5-base`
or `intfloat/multilingual-e5-large`. This module uses the current project model
to show the prefix pattern — swap the model name to use real E5.


In [2]:
"""
MODULE 2: DPR — Asymmetric Bi-Encoder Simulation

DPR Core Idea:
  - Query encoder E_Q: trained specifically on question-style text
  - Passage encoder E_P: trained specifically on answer/passage text
  - Trained with contrastive loss on (question, positive, negative) triplets

Modern practical equivalent: E5 with "query:" and "passage:" prefixes.

To use real E5, change MODEL_NAME to "intfloat/multilingual-e5-base"
and the prefixes will actually matter (E5 was trained on them).

With the current project model (paraphrase-multilingual-MiniLM-L12-v2)
prefixes have no semantic effect — this module shows WHERE to add them
so your code is ready to switch to E5.
"""

# Current project model — symmetric (no prefix effect)
SYMMETRIC_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

# To use real asymmetric DPR-style model, change to:
# ASYMMETRIC_MODEL = "intfloat/multilingual-e5-base"
# and add prefixes as shown below

print("Loading symmetric model (current project)...")
sym_model = SentenceTransformer(SYMMETRIC_MODEL)
print(f"Loaded: {SYMMETRIC_MODEL}")


def embed_symmetric(texts: list) -> np.ndarray:
    """Current project approach: same model for both queries and passages."""
    return sym_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)


def embed_with_prefixes(query: str = None, passages: list = None) -> tuple:
    """
    E5-style asymmetric encoding: different prefixes for queries vs passages.

    With paraphrase-multilingual-MiniLM-L12-v2:
      - prefixes have NO effect (model was not trained on them)
      - scores will be same as symmetric

    With intfloat/multilingual-e5-base or e5-large:
      - prefixes are REQUIRED and materially change the embeddings
      - query: prefix → E_Q behaviour
      - passage: prefix → E_P behaviour
    """
    results = {}
    if query is not None:
        prefixed_query = f"query: {query}"
        results["query_vec"] = sym_model.encode(prefixed_query, normalize_embeddings=True, show_progress_bar=False)
    if passages is not None:
        prefixed_passages = [f"passage: {p}" for p in passages]
        results["passage_vecs"] = sym_model.encode(prefixed_passages, normalize_embeddings=True, show_progress_bar=False)
    return results


# -----------------------------------------------------------------------
# Demo 1: Symmetric vs E5-style prefix comparison
# -----------------------------------------------------------------------
TEST_QUERY = "What is the penalty for premature FD withdrawal?"

print("\n" + "=" * 65)
print("DEMO 1: Symmetric vs Prefixed Encoding")
print("=" * 65)
print(f"Query: '{TEST_QUERY}'\n")

# Symmetric: no prefix
sym_query_vec  = embed_symmetric([TEST_QUERY])[0]
sym_passage_vecs = embed_symmetric(CORPUS_TEXTS)

# Prefixed: query: / passage: prefixes (effective with E5, ignored here)
prefixed = embed_with_prefixes(query=TEST_QUERY, passages=CORPUS_TEXTS)
pre_query_vec    = prefixed["query_vec"]
pre_passage_vecs = prefixed["passage_vecs"]

print("Symmetric (no prefix) scores:")
sym_scores = [cosine_sim(sym_query_vec, pv) for pv in sym_passage_vecs]
for i, (score, doc) in enumerate(sorted(zip(sym_scores, KNOWLEDGE_BASE), key=lambda x: x[0], reverse=True)):
    print(f"  Rank {i+1} | {score:.4f} | [{doc['doc_type'].upper():<7}] {doc['text'][:55]}...")

print("\nPrefixed (query:/passage:) scores [with this model, same result]:")
pre_scores = [cosine_sim(pre_query_vec, pv) for pv in pre_passage_vecs]
for i, (score, doc) in enumerate(sorted(zip(pre_scores, KNOWLEDGE_BASE), key=lambda x: x[0], reverse=True)):
    print(f"  Rank {i+1} | {score:.4f} | [{doc['doc_type'].upper():<7}] {doc['text'][:55]}...")

print("\nWith paraphrase-multilingual-MiniLM-L12-v2: scores are identical")
print("because this model was NOT trained with query:/passage: prefixes.")
print("Swap to intfloat/multilingual-e5-base for real DPR-style asymmetry.")

# -----------------------------------------------------------------------
# Demo 2: Why asymmetry matters — show the structural difference
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("DEMO 2: Query vs Passage Structural Difference")
print("=" * 65)

sample_query   = "What is the penalty for premature FD withdrawal?"
sample_passage = "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate."

q_words = set(sample_query.lower().split())
p_words = set(sample_passage.lower().split())
overlap = q_words & p_words
unique_q = q_words - p_words
unique_p = p_words - q_words

print(f"\nQuery:   '{sample_query}'")
print(f"Passage: '{sample_passage}'")
print(f"\nWord overlap (query ∩ passage): {overlap}")
print(f"Query-only words (interrogative): {unique_q - {'?', 'is', 'the', 'for', 'what'}}")
print(f"Passage-only words (factual):     {unique_p - {'a', 'on', 'the', 'of', 'its'}}")
print("\nConclusion: queries are interrogative + incomplete, passages are declarative + complete.")
print("A symmetric encoder learns a compromise. DPR trains two specialised encoders.")
print("Practical equivalent: E5 with 'query:' and 'passage:' prefixes — no training needed.")
print("\nCode pattern for E5 (change model and this works correctly):")
print("  query_vec   = model.encode('query: ' + query, normalize_embeddings=True)")
print("  passage_vecs = model.encode(['passage: ' + p for p in passages], normalize_embeddings=True)")
print("\nModule 2 complete. Run Module 3.")


Loading symmetric model (current project)...
Loaded: paraphrase-multilingual-MiniLM-L12-v2

DEMO 1: Symmetric vs Prefixed Encoding
Query: 'What is the penalty for premature FD withdrawal?'

Symmetric (no prefix) scores:
  Rank 1 | 0.7448 | [FAQ    ] Premature withdrawal of FD incurs a 1 percent penalty o...
  Rank 2 | 0.7058 | [SOP    ] This SOP covers Fixed Deposit premature withdrawal. Ste...
  Rank 3 | 0.5293 | [POLICY ] Fixed Deposit premature closure is allowed subject to a...
  Rank 4 | 0.4590 | [POLICY ] Early exit from your deposit account will attract a ded...
  Rank 5 | 0.3664 | [PRODUCT] The Fixed Deposit interest rate for 24 months is 7.1 pe...
  Rank 6 | 0.2980 | [PRODUCT] Senior citizens receive an additional 0.5 percent inter...

Prefixed (query:/passage:) scores [with this model, same result]:
  Rank 1 | 0.7807 | [FAQ    ] Premature withdrawal of FD incurs a 1 percent penalty o...
  Rank 2 | 0.7068 | [SOP    ] This SOP covers Fixed Deposit premature withdrawal. Ste...
 

## Module 3: ColBERT — MaxSim Late Interaction

ColBERT keeps one vector per token instead of one vector per document.
Scoring uses MaxSim: for each query token, find the maximum similarity
with any document token, then sum those maxima.

This module implements MaxSim from scratch to make the mechanics transparent.

**Why this matters for your project:**
Queries like "premature withdrawal" can match documents saying "early closure"
token-by-token: "premature" → "early", "withdrawal" → "closure" individually,
even if the single-vector embeddings blur these into one inexpressive average.


In [3]:
"""
MODULE 3: ColBERT — MaxSim Late Interaction

ColBERT formula:
  score(q, d) = Σ max cos_sim(E(q_i), E(d_j))
                 i   j

  For each query token q_i: find the document token d_j most similar to it.
  Score = sum of those maximum similarities across all query tokens.

This is more expressive than single-vector retrieval because:
  - Each query concept (premature, withdrawal, penalty) independently
    finds its best match in the document
  - A single-vector blends all concepts — if the blend does not happen
    to be close to the document's single-vector blend, it misses
"""

print("Loading model for ColBERT simulation...")
# Using the same model for both query and document encoding (simplified ColBERT)
# Real ColBERT has a dedicated linear projection layer on top of BERT
# and uses a smaller per-token dimension (128-dim) vs BERT's full 768-dim
colbert_model = SentenceTransformer(SYMMETRIC_MODEL)
print(f"Loaded: {SYMMETRIC_MODEL}")


def encode_tokens(text: str, model: SentenceTransformer) -> np.ndarray:
    """
    Simulate ColBERT token-level encoding.

    Real ColBERT: run BERT, take each token's hidden state, apply linear projection to 128-dim.
    Here: encode each word individually as a proxy for token-level vectors.

    This is a SIMPLIFICATION for illustration — real ColBERT encodes the
    full sequence and uses BERT's contextual token representations,
    not individual word embeddings.
    """
    words = text.lower().split()
    if not words:
        return np.zeros((1, 384))
    # Encode each word with its position context (single word = decontextualised, but illustrative)
    word_vecs = model.encode(words, normalize_embeddings=True, show_progress_bar=False)
    return word_vecs  # shape: (num_words, 384)


def maxsim_score(query_vecs: np.ndarray, doc_vecs: np.ndarray) -> float:
    """
    ColBERT MaxSim scoring.

    For each query token vector:
      - Compute cosine similarity with all document token vectors
      - Take the maximum similarity
    Sum those maxima across all query tokens.

    query_vecs: shape (num_query_tokens, dim)
    doc_vecs:   shape (num_doc_tokens, dim)
    """
    total = 0.0
    for q_vec in query_vecs:
        # Similarity between this query token and all document tokens
        sims = [cosine_sim(q_vec, d_vec) for d_vec in doc_vecs]
        total += max(sims)  # MaxSim: best-matching document token for this query token
    return total


def single_vector_score(query: str, doc_text: str, model: SentenceTransformer) -> float:
    """Standard single-vector cosine similarity (current project approach)."""
    vecs = model.encode([query, doc_text], normalize_embeddings=True, show_progress_bar=False)
    return cosine_sim(vecs[0], vecs[1])


# -----------------------------------------------------------------------
# Demo: MaxSim vs single-vector on vocabulary mismatch pairs
# -----------------------------------------------------------------------
TEST_PAIRS = [
    ("premature withdrawal penalty",
     "Premature withdrawal of FD incurs a 1 percent penalty.",
     "exact match — both methods should agree"),
    ("early closure charges fixed deposit",
     "Premature withdrawal of FD incurs a 1 percent penalty.",
     "paraphrase — single-vector may miss, MaxSim should find token matches"),
    ("exit deposit fee deduction",
     "Early exit from your deposit account will attract a deduction from your returns.",
     "vocabulary overlap — should match on deposit, exit, deduction"),
    ("FD band karna penalty",
     "Premature withdrawal of FD incurs a 1 percent penalty.",
     "Hinglish — FD and penalty overlap, band karna is a mismatch"),
]

print("\n" + "=" * 65)
print("MAXSIM vs SINGLE-VECTOR — VOCABULARY MISMATCH PAIRS")
print("=" * 65)

for query, passage, description in TEST_PAIRS:
    # Encode query and passage tokens
    q_vecs = encode_tokens(query, colbert_model)
    d_vecs = encode_tokens(passage, colbert_model)

    # MaxSim score
    ms_score  = maxsim_score(q_vecs, d_vecs)

    # Single-vector score
    sv_score  = single_vector_score(query, passage, colbert_model)

    print(f"\n  Query:   '{query}'")
    print(f"  Passage: '{passage[:60]}...'")
    print(f"  Case:    {description}")
    print(f"  MaxSim score:        {ms_score:.4f}")
    print(f"  Single-vector score: {sv_score:.4f}")
    print(f"  MaxSim {'HIGHER' if ms_score > sv_score else 'LOWER or EQUAL'} (expected: usually higher for paraphrase cases)")

# -----------------------------------------------------------------------
# Show token-level matching (what MaxSim actually does)
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("INSIDE MAXSIM: Token-Level Matching (Illustrative)")
print("=" * 65)

query   = "premature withdrawal penalty"
doc     = "early closure charges on FD account"
q_words = query.split()
d_words = doc.split()

q_vecs = encode_tokens(query, colbert_model)
d_vecs = encode_tokens(doc, colbert_model)

print(f"\nQuery:    '{query}'")
print(f"Document: '{doc}'")
print()

total = 0.0
for i, (q_word, q_vec) in enumerate(zip(q_words, q_vecs)):
    sims = [(cosine_sim(q_vec, d_vec), d_word) for d_vec, d_word in zip(d_vecs, d_words)]
    best_sim, best_word = max(sims, key=lambda x: x[0])
    total += best_sim
    print(f"  Query token '{q_word}' → best match in doc: '{best_word}' (sim={best_sim:.4f})")

print(f"\n  MaxSim total = {total:.4f}")
print(f"\n  This is why ColBERT catches paraphrase matches:")
print(f"  'premature' → 'closure' (both = stopping early)")
print(f"  'withdrawal' → 'charges' (related concepts in FD context)")
print(f"  'penalty' → 'charges' (near-synonyms)")
print(f"  Single-vector blends all three → may miss the match entirely.")
print("\nNote: This simulation uses decontextualised word embeddings.")
print("Real ColBERT uses BERT's contextual token representations — better quality.")
print("Use ragatouille for production ColBERT v2.")
print("\nModule 3 complete. Run Module 4.")


Loading model for ColBERT simulation...
Loaded: paraphrase-multilingual-MiniLM-L12-v2

MAXSIM vs SINGLE-VECTOR — VOCABULARY MISMATCH PAIRS

  Query:   'premature withdrawal penalty'
  Passage: 'Premature withdrawal of FD incurs a 1 percent penalty....'
  Case:    exact match — both methods should agree
  MaxSim score:        2.9745
  Single-vector score: 0.7270
  MaxSim HIGHER (expected: usually higher for paraphrase cases)

  Query:   'early closure charges fixed deposit'
  Passage: 'Premature withdrawal of FD incurs a 1 percent penalty....'
  Case:    paraphrase — single-vector may miss, MaxSim should find token matches
  MaxSim score:        2.8131
  Single-vector score: 0.4248
  MaxSim HIGHER (expected: usually higher for paraphrase cases)

  Query:   'exit deposit fee deduction'
  Passage: 'Early exit from your deposit account will attract a deductio...'
  Case:    vocabulary overlap — should match on deposit, exit, deduction
  MaxSim score:        3.5237
  Single-vector score: 0.

## Module 4: SPLADE — Learned Sparse Expansion Simulation

SPLADE uses a transformer MLM head to expand documents into learned sparse vectors
over the full vocabulary. This bridges vocabulary mismatch at the sparse (BM25) layer.

This module simulates SPLADE vocabulary expansion by:
1. Showing the concept of vocabulary expansion
2. Simulating what expanded BM25 matching would look like
3. Demonstrating how SPLADE bridges Hinglish → English

**For production SPLADE:** use `naver-splade-v3` or `splade-cocondenser-ensembledistil`
from Hugging Face with the `splade` library.


In [4]:
"""
MODULE 4: SPLADE — Learned Sparse Expansion Simulation

SPLADE formula:
  w_vocab(d) = max_pooling_over_tokens( log(1 + ReLU( MLM_head(BERT(d)) ) ) )

This produces a sparse vector over the full vocabulary (~30k terms) where
non-zero dimensions correspond to terms the document "expands" to based on
learned co-occurrence patterns.

This simulation uses a manually curated expansion dictionary to illustrate
the concept. Real SPLADE learns these expansions from retrieval training data.
"""

from rank_bm25 import BM25Okapi

# -----------------------------------------------------------------------
# Simulated SPLADE vocabulary expansion dictionary
# Built from domain knowledge about your FD corpus:
#   - Hinglish to English mappings
#   - Synonym mappings learned from co-occurrence
#   - Domain-specific implicit terms
#
# Real SPLADE learns these from (query, passage) training pairs.
# This manual dictionary illustrates what SPLADE would learn.
# -----------------------------------------------------------------------
SPLADE_EXPANSION = {
    # Document terms → implicit expanded terms
    "premature":    ["early", "closure", "exit", "before", "nikalna", "band"],
    "withdrawal":   ["closure", "exit", "deduction", "forfeit", "nikalna", "band"],
    "penalty":      ["charges", "fee", "deduction", "fine", "cut"],
    "interest":     ["byaj", "rate", "return", "earning", "profit"],
    "maturity":     ["machurity", "due", "date", "proceeds", "payout", "credited"],
    "FD":           ["fixed deposit", "deposit", "account", "scheme", "jama"],
    "closure":      ["premature", "early", "exit", "withdrawal", "band"],
    "early":        ["premature", "before", "advance", "closure"],
    "exit":         ["withdrawal", "closure", "premature", "nikalna"],
    # Hinglish → English mappings (what SPLADE would learn for your corpus)
    "byaj":         ["interest", "rate", "return"],
    "machurity":    ["maturity", "due date", "proceeds"],
    "nikalna":      ["withdrawal", "exit", "closure", "premature"],
    "band":         ["closure", "close", "stop", "premature"],
    "paisa":        ["amount", "money", "funds", "proceeds"],
}


def expand_text(text: str, expansion_dict: dict, weight: float = 0.5) -> str:
    """
    Simulate SPLADE vocabulary expansion.
    For each word in the text, adds related terms from the expansion dictionary.

    Real SPLADE: the expanded terms get learned weights (not all equal).
    Here: all expansions get the same weight (a simplification).

    weight: controls how many expanded terms to add (0-1 fraction)
    """
    words     = text.lower().split()
    expanded  = list(words)  # start with original words

    for word in words:
        if word in expansion_dict:
            additions = expansion_dict[word]
            # Add all expansions (real SPLADE only adds terms above a threshold)
            for term in additions:
                if term not in expanded:
                    expanded.append(term)

    return " ".join(expanded)


def tokenize(text: str) -> list:
    return text.lower().split()


# -----------------------------------------------------------------------
# Build two BM25 indexes:
# 1. Standard BM25 (no expansion) — what you have now
# 2. SPLADE-style BM25 (with expansion) — what SPLADE provides
# -----------------------------------------------------------------------

# Standard BM25
standard_corpus = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
bm25_standard   = BM25Okapi(standard_corpus, k1=1.2, b=0.75)

# SPLADE-style expanded BM25
expanded_corpus = [tokenize(expand_text(doc["text"], SPLADE_EXPANSION)) for doc in KNOWLEDGE_BASE]
bm25_expanded   = BM25Okapi(expanded_corpus, k1=1.2, b=0.75)

print("=" * 65)
print("SPLADE VOCABULARY EXPANSION — SIMULATION")
print("=" * 65)

# Show what expansion does to a document
sample_doc = KNOWLEDGE_BASE[0]["text"]
sample_expanded = expand_text(sample_doc, SPLADE_EXPANSION)
print(f"\nOriginal:  '{sample_doc}'")
print(f"Expanded:   '{sample_expanded}'")
print("\nExpansion adds implicit terms: closure, exit, charges, fee, etc.")
print("These are terms the document IMPLIES but never states.")

# -----------------------------------------------------------------------
# Demo: Standard vs SPLADE-expanded BM25 on Hinglish queries
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("STANDARD BM25 vs SPLADE-EXPANDED BM25 ON HINGLISH QUERIES")
print("=" * 65)

hinglish_queries = [
    ("machurity ka paisa kab aayega",           "Hinglish maturity query"),
    ("FD band karna hai penalty kitni hogi",    "Hinglish premature closure query"),
    ("byaj wala FD nikalna hai",                "Hinglish withdrawal with byaj"),
    ("early closure charges fixed deposit",     "English paraphrase"),
    ("premature withdrawal penalty FD",         "English exact terms"),
]

for query, label in hinglish_queries:
    q_tokens     = tokenize(query)
    q_expanded   = tokenize(expand_text(query, SPLADE_EXPANSION))

    std_scores   = bm25_standard.get_scores(q_tokens)
    exp_scores   = bm25_expanded.get_scores(q_expanded)

    std_best_idx  = int(np.argmax(std_scores))
    exp_best_idx  = int(np.argmax(exp_scores))

    print(f"\n  [{label}]")
    print(f"  Query: '{query}'")
    print(f"  Expanded query: '{expand_text(query, SPLADE_EXPANSION)}'")
    print(f"  Standard BM25  → all zero: {all(s == 0 for s in std_scores)} | max score: {max(std_scores):.3f}")
    print(f"  Expanded BM25  → all zero: {all(s == 0 for s in exp_scores)} | max score: {max(exp_scores):.3f}")
    if max(exp_scores) > 0:
        print(f"  Best match:  '{KNOWLEDGE_BASE[exp_best_idx]['text'][:60]}...'")

# -----------------------------------------------------------------------
# Summary: what SPLADE learns vs what BM25 does
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("SPLADE vs BM25 — KEY DIFFERENCES")
print("=" * 65)
print("""
BM25 (standard):
  - Index document terms as-is
  - Query matched against exact document vocabulary
  - "byaj" query scores 0 against "interest" document → vocabulary mismatch
  - Fast (inverted index), interpretable, free

SPLADE (learned sparse expansion):
  - At index time: transformer MLM head generates expanded vocabulary weights
  - "interest" document gets non-zero weight for "byaj" (learned from training data)
  - Query "byaj" now matches the expanded document representation
  - Same inverted index infrastructure at query time (millisecond latency)
  - GPU needed at index time (not query time)
  - Weights are learned from (query, passage) training pairs

For your project (64.4% Hinglish emails, English policy docs):
  SPLADE would learn:
    byaj      ↔ interest  (financial context)
    machurity ↔ maturity  (spelling variant)
    nikalna   ↔ withdrawal (semantic equivalent)
    band      ↔ closure    (semantic equivalent)
  Without any manual synonym dictionary.

Production options:
  pip install splade
  model: naver-splade-v3 or splade-cocondenser-ensembledistil (HuggingFace)
  Infrastructure: Qdrant sparse vectors or Elasticsearch sparse_vector
""")
print("Module 4 complete. Run Module 5.")


SPLADE VOCABULARY EXPANSION — SIMULATION

Original:  'Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.'
Expanded:   'premature withdrawal of fd incurs a 1 percent penalty on the applicable interest rate. early closure exit before nikalna band deduction forfeit charges fee fine cut byaj rate return earning profit'

Expansion adds implicit terms: closure, exit, charges, fee, etc.
These are terms the document IMPLIES but never states.

STANDARD BM25 vs SPLADE-EXPANDED BM25 ON HINGLISH QUERIES

  [Hinglish maturity query]
  Query: 'machurity ka paisa kab aayega'
  Expanded query: 'machurity ka paisa kab aayega maturity due date proceeds amount money funds'
  Standard BM25  → all zero: True | max score: 0.000
  Expanded BM25  → all zero: True | max score: 0.000

  [Hinglish premature closure query]
  Query: 'FD band karna hai penalty kitni hogi'
  Expanded query: 'fd band karna hai penalty kitni hogi closure close stop premature charges fee deduction fi

## Module 5: Head-to-Head Comparison and Upgrade Path

Compares all methods on the same test queries.
Provides a concrete decision framework for when to use each architecture.


In [5]:
"""
MODULE 5: Head-to-Head Comparison and Upgrade Path

Runs all four retrieval approaches on the same query variations:
  1. BM25 (standard, Topic 1)
  2. Dense single-vector (current project, Topic 2)
  3. E5-prefix (DPR-style asymmetric, Module 2)
  4. MaxSim (ColBERT-style, Module 3)
  5. SPLADE-expanded BM25 (Module 4)

Shows where each method wins and loses.
"""

from rank_bm25 import BM25Okapi
import numpy as np


def tokenize(text: str) -> list:
    return text.lower().split()


def expand_text(text: str, expansion_dict: dict) -> str:
    words    = text.lower().split()
    expanded = list(words)
    for word in words:
        for term in expansion_dict.get(word, []):
            if term not in expanded:
                expanded.append(term)
    return " ".join(expanded)


# SPLADE expansion dict (from Module 4)
SPLADE_EXPANSION = {
    "premature": ["early", "closure", "exit", "nikalna", "band"],
    "withdrawal": ["closure", "exit", "deduction", "nikalna", "band"],
    "penalty": ["charges", "fee", "deduction", "fine"],
    "interest": ["byaj", "rate", "return"],
    "maturity": ["machurity", "due", "proceeds", "payout"],
    "FD": ["fixed deposit", "deposit", "account"],
    "closure": ["premature", "early", "exit", "withdrawal"],
    "early": ["premature", "before", "closure"],
    "byaj": ["interest", "rate", "return"],
    "machurity": ["maturity", "proceeds"],
    "nikalna": ["withdrawal", "exit", "closure"],
    "band": ["closure", "close", "premature"],
}


# Build indexes
standard_corpus = [tokenize(doc["text"]) for doc in KNOWLEDGE_BASE]
expanded_corpus = [tokenize(expand_text(doc["text"], SPLADE_EXPANSION)) for doc in KNOWLEDGE_BASE]
bm25_std  = BM25Okapi(standard_corpus, k1=1.2, b=0.75)
bm25_exp  = BM25Okapi(expanded_corpus, k1=1.2, b=0.75)

# Dense model (current project)
dense_model = SentenceTransformer(SYMMETRIC_MODEL)

def dense_scores(query: str) -> list:
    q_vec  = dense_model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    p_vecs = dense_model.encode(CORPUS_TEXTS, normalize_embeddings=True, show_progress_bar=False)
    return [float(np.dot(q_vec, pv)) for pv in p_vecs]


def maxsim_scores_all(query: str) -> list:
    """MaxSim for all documents (simplified word-level)."""
    q_vecs = dense_model.encode(query.split(), normalize_embeddings=True, show_progress_bar=False)
    scores = []
    for doc in KNOWLEDGE_BASE:
        d_vecs = dense_model.encode(doc["text"].split(), normalize_embeddings=True, show_progress_bar=False)
        total = sum(max(float(np.dot(qv, dv)) for dv in d_vecs) for qv in q_vecs)
        scores.append(total)
    return scores


# -----------------------------------------------------------------------
# Comparison table across query variations
# -----------------------------------------------------------------------
def get_top1_doc(scores: list) -> dict:
    return KNOWLEDGE_BASE[int(np.argmax(scores))]


print("=" * 65)
print("HEAD-TO-HEAD: TOP-1 DOCUMENT ACROSS ALL METHODS")
print("=" * 65)

TEST_QUERIES = [
    ("English exact",     "premature withdrawal penalty FD"),
    ("English paraphrase","early closure charges fixed deposit"),
    ("Hinglish maturity", "machurity ka paisa kab aayega"),
    ("Hinglish premature","FD band karna penalty kitni"),
    ("Hinglish byaj",     "byaj wala FD nikalna hai"),
]

for label, query in TEST_QUERIES:
    # Scores from each method
    bm25_s   = bm25_std.get_scores(tokenize(query))
    exp_s    = bm25_exp.get_scores(tokenize(expand_text(query, SPLADE_EXPANSION)))
    dense_s  = dense_scores(query)
    ms_s     = maxsim_scores_all(query)

    # Top-1 document for each method
    bm25_top1   = KNOWLEDGE_BASE[int(np.argmax(bm25_s))]   if max(bm25_s) > 0 else None
    exp_top1    = KNOWLEDGE_BASE[int(np.argmax(exp_s))]    if max(exp_s) > 0 else None
    dense_top1  = KNOWLEDGE_BASE[int(np.argmax(dense_s))]
    ms_top1     = KNOWLEDGE_BASE[int(np.argmax(ms_s))]

    print(f"\n[{label}] Query: '{query}'")
    print(f"  BM25 standard:    {f'{bm25_top1["doc_type"].upper()} | {bm25_top1["text"][:45]}...' if bm25_top1 else 'ALL ZERO'}")
    print(f"  SPLADE-expanded:  {f'{exp_top1["doc_type"].upper()} | {exp_top1["text"][:45]}...' if exp_top1 else 'ALL ZERO'}")
    print(f"  Dense (current):  {dense_top1['doc_type'].upper()} | {dense_top1['text'][:45]}...")
    print(f"  MaxSim (ColBERT): {ms_top1['doc_type'].upper()} | {ms_top1['text'][:45]}...")

# -----------------------------------------------------------------------
# Decision framework
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("UPGRADE DECISION FRAMEWORK FOR YOUR PROJECT")
print("=" * 65)
print("""
Step 0 (do this first): Hybrid BM25 + dense + RRF (Topic 4)
  - No training, no new infrastructure, no re-embedding
  - BM25 handles exact matches, dense handles vocabulary gaps
  - Often achieves 80-90% of quality improvement at zero training cost
  - Measure Recall@K before doing anything else

Step 1 (if hybrid is insufficient — embedding model upgrade):
  - Replace paraphrase-multilingual-MiniLM-L12-v2 with intfloat/multilingual-e5-base
  - Add "query: " prefix to queries, "passage: " prefix to passages
  - Re-embed 17 pages (trivial). Measure Recall@K again.
  - Cost: 0 (free model, 5 minutes of work)

Step 2 (if Hinglish vocabulary mismatch persists — SPLADE):
  - Use naver-splade-v3 (multilingual) from HuggingFace
  - Index your 17 pages with SPLADE (GPU recommended, your RTX 4060 works)
  - Query expansion using SPLADE query encoder
  - Infrastructure: Qdrant sparse vectors OR Elasticsearch sparse_vector field
  - Cost: few hours of setup, no training needed if using pre-trained model

Step 3 (if fine-grained matching is the bottleneck — ColBERT):
  - Use ragatouille (4 lines of code, handles everything)
  - Indexes 17 pages in seconds on your RTX 4060
  - Query in milliseconds
  - Cost: pip install ragatouille, 30 minutes of setup

Step 4 (if domain-specific quality is still insufficient — fine-tuning):
  - Create 200-500 (Hinglish email, policy chunk) training pairs
  - Fine-tune E5 or SPLADE on these domain-specific pairs
  - Biggest quality jump but highest cost
  - Covered in Chapter 18 (Fine-Tuning)
""")
print("Module 5 complete. All Topic 3 modules done.")
print("Next: Topic 4 — Hybrid Search with BM25 + Dense + RRF")


HEAD-TO-HEAD: TOP-1 DOCUMENT ACROSS ALL METHODS

[English exact] Query: 'premature withdrawal penalty FD'
  BM25 standard:    FAQ | Premature withdrawal of FD incurs a 1 percent...
  SPLADE-expanded:  FAQ | Premature withdrawal of FD incurs a 1 percent...
  Dense (current):  FAQ | Premature withdrawal of FD incurs a 1 percent...
  MaxSim (ColBERT): SOP | This SOP covers Fixed Deposit premature withd...

[English paraphrase] Query: 'early closure charges fixed deposit'
  BM25 standard:    POLICY | Fixed Deposit premature closure is allowed su...
  SPLADE-expanded:  POLICY | Early exit from your deposit account will att...
  Dense (current):  POLICY | Fixed Deposit premature closure is allowed su...
  MaxSim (ColBERT): POLICY | Fixed Deposit premature closure is allowed su...

[Hinglish maturity] Query: 'machurity ka paisa kab aayega'
  BM25 standard:    ALL ZERO
  SPLADE-expanded:  ALL ZERO
  Dense (current):  SOP | This SOP covers Fixed Deposit premature withd...
  MaxSim (ColBERT): PO